# 03 — Geospatial Structure & Baseline Accessibility

**Objective:** Establish the spatial understanding and baseline accessibility metrics
that will serve as inputs for the optimization pipeline.

This notebook:
1. Computes province-level response time statistics (P50, P90)
2. Quantifies the nearest-AED distance distribution per province
3. Visualizes the spatial structure of missions, AEDs, and high-delay events

In [ ]:
from pathlib import Path
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np

PROJECT_ROOT = Path('..')
V3_DIR = PROJECT_ROOT / 'mda_project' / 'data' / 'processed_v3'
RAW_DIR = PROJECT_ROOT / 'mda_project' / 'data' / 'raw'

mission = pd.read_parquet(V3_DIR / 'mission_records_v3.parquet')
aed = pd.read_parquet(V3_DIR / 'aed_records_v3.parquet')
boundary = gpd.read_file(RAW_DIR / 'BELGIUM_-_Provinces.geojson')

print(f'Missions: {len(mission):,} | AEDs: {len(aed):,} | Provinces: {len(boundary)}')

## 3.1 Province-Level Response Statistics

We compute the median (P50) and 90th-percentile (P90) response times and nearest-AED distances
per province to identify regional disparities.

In [ ]:
prov = mission.groupby('province').agg(
    n_missions=('mission_id', 'count'),
    response_p50=('response_min', 'median'),
    response_p90=('response_min', lambda s: s.quantile(0.9)),
    dist_aed_p50=('dist_to_aed_km', 'median'),
    dist_aed_p90=('dist_to_aed_km', lambda s: s.quantile(0.9)),
).sort_values('response_p90', ascending=False)

display(prov.style.format({'response_p50': '{:.1f}', 'response_p90': '{:.1f}',
                           'dist_aed_p50': '{:.2f}', 'dist_aed_p90': '{:.2f}',
                           'n_missions': '{:,}'}))

## 3.2 Response Time Comparison by Province

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

prov_sorted = prov.sort_values('response_p90')
y_pos = range(len(prov_sorted))

ax.barh(y_pos, prov_sorted['response_p90'], color='#e74c3c', alpha=0.8, label='P90 Response Time')
ax.barh(y_pos, prov_sorted['response_p50'], color='#3498db', alpha=0.8, label='P50 Response Time')

ax.set_yticks(y_pos)
ax.set_yticklabels(prov_sorted.index)
ax.set_xlabel('Response Time [min]', fontsize=12)
ax.set_title('Emergency Response Times by Province', fontsize=14, fontweight='bold')
ax.legend(loc='lower right')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

## 3.3 Spatial Structure Map

In [ ]:
fig, ax = plt.subplots(figsize=(10, 9), dpi=140)
boundary.to_crs(4326).boundary.plot(ax=ax, color='black', linewidth=0.6)

# All missions (light background)
ax.scatter(mission['longitude'], mission['latitude'],
           s=1, alpha=0.06, color='#90a4ae', label='All Missions', rasterized=True)

# Top 10% delayed events (red highlight)
threshold = mission['response_min'].quantile(0.9)
high = mission[mission['response_min'] >= threshold]
ax.scatter(high['longitude'], high['latitude'],
           s=2, alpha=0.25, color='#ef476f', label=f'Top 10% Delay (>={threshold:.0f} min)')

# Existing AEDs
ax.scatter(aed['longitude'], aed['latitude'],
           s=4, alpha=0.4, color='#06d6a0', marker='^', label=f'AED Locations (n={len(aed):,})')

ax.legend(loc='lower left', fontsize=10, markerscale=4)
ax.set_title('Belgium: Mission Distribution, Delay Hotspots, and AED Coverage', fontsize=13, fontweight='bold')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
plt.tight_layout()
plt.show()

## Summary

The geospatial baseline reveals:
- Significant inter-provincial disparities in P90 response times
- Dense AED coverage in urban Flanders vs. sparse coverage in rural Wallonia
- High-delay events cluster in specific sub-regions, motivating targeted facility optimization

These spatial patterns feed directly into **Notebook 04** (Predictive Modeling) and **Notebook 05** (Multi-Objective Optimization).